In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler 플러그인 설치 및 사용

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</details>
Qiskit 사용자 커뮤니티에서 커스텀 트랜스파일 코드의 개발과 재사용을 촉진하기 위해, Qiskit SDK는 서드파티 Python 패키지가 Qiskit을 통해 접근 가능한 확장 트랜스파일 기능을 제공한다고 선언할 수 있는 플러그인 인터페이스를 지원합니다.

현재 서드파티 플러그인은 다음 세 가지 방법으로 확장 트랜스파일 기능을 제공할 수 있습니다:

- [Transpiler 스테이지 플러그인](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins)은 프리셋 스테이지 패스 매니저의 [6가지 스테이지](transpiler-stages) 중 하나를 대체할 수 있는 패스 매니저를 제공합니다: `init`, `layout`, `routing`, `translation`, `optimization`, `scheduling`.
- [유니터리 합성 플러그인](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin)은 유니터리 Gate 합성을 위한 확장 기능을 제공합니다.
- [고수준 합성 플러그인](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin)은 선형 함수나 Clifford 연산자와 같은 "고수준 객체" 합성을 위한 확장 기능을 제공합니다. 고수준 객체는 [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) 클래스의 서브클래스로 표현됩니다.

이 페이지의 나머지 부분에서는 사용 가능한 플러그인을 나열하고, 새 플러그인을 설치하고, 사용하는 방법을 설명합니다.

## 사용 가능한 플러그인 목록 조회 및 새 플러그인 설치
Qiskit에는 이미 트랜스파일을 위한 기본 내장 플러그인이 포함되어 있습니다. 추가 플러그인을 설치하려면 Python 패키지 매니저를 사용하면 됩니다. 예를 들어, `pip install qiskit-toqm`을 실행하면 [Qiskit TOQM](https://github.com/qiskit-toqm/qiskit-toqm) 라우팅 스테이지 플러그인을 설치할 수 있습니다. 다양한 서드파티 플러그인이 [Qiskit 에코시스템](https://qiskit.github.io/ecosystem/#transpiler_plugin)의 일부로 제공됩니다.

### 사용 가능한 Transpiler 스테이지 플러그인 목록 조회
[list_stage_plugins](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#qiskit.transpiler.preset_passmanagers.plugin.list_stage_plugins) 함수를 사용하고, 목록을 조회할 스테이지 이름을 인수로 전달합니다.

In [1]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

In [2]:
list_stage_plugins("routing")

['basic', 'default', 'lookahead', 'none', 'sabre']

 If `qiskit-toqm` were installed, then `toqm` would appear in the list of `routing` plugins.

### List available unitary synthesis plugins

Use the [unitary_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names) function.

In [3]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

### List available high-level synthesis plugins

Use the [high_level_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names) function, passing the name of the type of "high-level object" to be synthesized. The name corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class representing the type of object being synthesized.

In [4]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

`qiskit-toqm`이 설치되어 있다면, `toqm`이 `routing` 플러그인 목록에 표시됩니다.

### 사용 가능한 유니터리 합성 플러그인 목록 조회
[unitary_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names) 함수를 사용합니다.

In [5]:
from qiskit.transpiler.passes.synthesis.plugin import (
    HighLevelSynthesisPluginManager,
)

HighLevelSynthesisPluginManager().plugins.names()

['FullAdder.default',
 'FullAdder.ripple_c04',
 'FullAdder.ripple_v95',
 'HalfAdder.default',
 'HalfAdder.qft_d00',
 'HalfAdder.ripple_c04',
 'HalfAdder.ripple_r25',
 'HalfAdder.ripple_v95',
 'IntComp.default',
 'IntComp.noaux',
 'IntComp.twos',
 'ModularAdder.default',
 'ModularAdder.modular_v17',
 'ModularAdder.qft_d00',
 'ModularAdder.ripple_c04',
 'ModularAdder.ripple_v95',
 'Multiplier.cumulative_h18',
 'Multiplier.default',
 'Multiplier.qft_r17',
 'PauliEvolution.default',
 'PauliEvolution.rustiq',
 'WeightedSum.default',
 'annotated.default',
 'clifford.ag',
 'clifford.bm',
 'clifford.default',
 'clifford.greedy',
 'clifford.layers',
 'clifford.lnn',
 'linear_function.default',
 'linear_function.kms',
 'linear_function.pmh',
 'mcmt.default',
 'mcmt.noaux',
 'mcmt.vchain',
 'mcmt.xgate',
 'mcx.1_clean_b95',
 'mcx.1_clean_kg24',
 'mcx.1_dirty_kg24',
 'mcx.2_clean_kg24',
 'mcx.2_dirty_kg24',
 'mcx.default',
 'mcx.gray_code',
 'mcx.n_clean_m15',
 'mcx.n_dirty_i15',
 'mcx.noaux_hp24'

## Use a plugin

In this section, we show how to use transpiler plugins. In the code examples, we use plugins that come with Qiskit, but plugins installed from third-party packages are used the same way.

### Use a transpiler stage plugin

To use a transpiler stage plugin, specify its name with the appropriate argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). The argument is formed by appending `_method` to the name of the transpilation stage. For example, to use the `lookahead` routing plugin, we would specify `lookahead` for the `routing_method` argument.

<Admonition type="note">
The  `FakeSherbrooke` mock backend from `qiskit_ibm_runtime` is used in these examples, but you can try it on any Qiskit-compatible real or fake backend.  Your results might be different.
</Admonition>

In [6]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, routing_method="lookahead"
)

### 사용 가능한 고수준 합성 플러그인 목록 조회
[high_level_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names) 함수를 사용하고, 합성할 "고수준 객체" 유형의 이름을 인수로 전달합니다. 이름은 합성 대상 객체 유형을 나타내는 [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) 클래스의 [`name`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation#name) 속성에 해당합니다.

In [7]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    unitary_synthesis_method="sk",
    unitary_synthesis_plugin_config=dict(
        basis_gates=["cz", "id", "rz", "sx", "x"]
    ),
)

Unitary synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

Use the `unitary_synthesis_plugin_config` argument, a free-form dictionary, to pass options for the unitary synthesis method. The documentation of the synthesis method should explain the options it supports. See [this list](/docs/api/qiskit/transpiler_synthesis_plugins#unitary-synthesis-plugins-2) for links to the documentation of the built-in unitary synthesis plugins.

### Use a high-level synthesis plugin

First, create an [HLSConfig](/docs/api/qiskit/qiskit.transpiler.passes.HLSConfig) to
store the names of the plugins to use for various high-level objects. For example:

In [8]:
from qiskit.transpiler.passes import HLSConfig

hls_config = HLSConfig(clifford=["layers"], linear_function=["pmh"])

[HighLevelSynthesisPluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPluginManager) 클래스를 사용하면 모든 고수준 합성 플러그인의 이름을 나열할 수 있습니다:

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, hls_config=hls_config
)

High-level synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

## Next steps

<Admonition type="tip" title="Recommendation">
    - [Create a transpiler plugin](/docs/guides/create-transpiler-plugin).
    - Check out the [tutorials](/docs/tutorials) for examples of transpiling and running quantum circuits.
</Admonition>